# 05 — Reduce Sum

## Background

**Reduction** combines a set of values into a single result (sum, max, min, etc.). It appears everywhere in neural networks: mean and variance in LayerNorm/BatchNorm, the sum of exponentials in Softmax, global loss aggregation.

The GPU implementation challenge is **cross-thread data dependency**: unlike element-wise ops, reduction requires inter-thread communication (warp shuffle or shared memory). The three frameworks handle this at different abstraction levels:

- **Triton**: one program per row; Python loop + `tl.sum` for intra-block reduction
- **TileLang**: `T.reduce_sum` + `T.Serial` for tiled iteration

## Problem Definition

Given matrix `A [N, M]`, compute the row-wise sum into vector `B [N]`:

$$B[i] = \sum_{j=0}^{M-1} A[i, j], \quad i = 0, 1, \ldots, N-1$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_reduce_sum(A):
    return torch.sum(A, dim=1)   # [N, M] → [N]

B_ref = ref_reduce_sum(A)
print(f"A: {A.shape} → B: {B_ref.shape}")

## Triton Implementation

One Triton program per row: load chunks of size `BLOCK_M` in a Python loop, accumulate into a `[BLOCK_M]` buffer, then `tl.sum(acc, 0)` reduces within the warp.

> `tl.sum(x, 0)` reduces a 1D tensor `x [BLOCK_M]`, returning a scalar (0-d Triton tensor).

In [ ]:
@triton.jit
def _reduce_sum_kernel(A_ptr, B_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)   # one program per row
    base = A_ptr + row * M
    acc  = tl.zeros([BLOCK_M], dtype=tl.float32)
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=0.0)
        acc   = acc + chunk.to(tl.float32)
    tl.store(B_ptr + row, tl.sum(acc, 0))

BLOCK_M_TRI = 1024

def triton_reduce_sum(A):
    N_, M_ = A.shape
    B = torch.empty(N_, dtype=A.dtype, device=A.device)
    _reduce_sum_kernel[(N_,)](A, B, M_, BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_reduce_sum(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-2)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.4f}")

## TileLang Implementation

`T.reduce_sum(src, dst, dim=1, clear=False)` wraps warp-level reduction; `T.Serial` signals a sequential loop (cross-iteration data dependency); `clear=False` means accumulate mode.

In [ ]:
# Optimised for gfx1100: BN=2 rows per block, BM=128 cols per tile, threads=128.
# Constraint: BM <= threads (T.copy layout), and BN*BM fits the AllReduce warp reduction.
# threads=128 → 4 wavefronts of 32 per block, good CU occupancy on RDNA3.
BN, BM = 2, 128

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_reduce_sum(A, BLOCK_N: int, BLOCK_M: int):
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_,), dtype)
    with T.Kernel(N_ // BLOCK_N, threads=128) as pid_n:
        A_local = T.alloc_fragment((BLOCK_N, BLOCK_M), dtype)
        B_local = T.alloc_fragment((BLOCK_N,), dtype)
        T.clear(B_local)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[pid_n * BLOCK_N, m_blk * BLOCK_M], A_local)
            T.reduce_sum(A_local, B_local, dim=1, clear=False)
        T.copy(B_local, B[pid_n * BLOCK_N])
    return B

k = tl_reduce_sum.compile(N=N, M=M, BLOCK_N=BN, BLOCK_M=BM)
B_tl = k(A.clone())
ok = torch.allclose(B_ref, B_tl, atol=1e-2)
print("TileLang correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.4f}")

## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch": bench(ref_reduce_sum,    [A]),
    "Triton":  bench(triton_reduce_sum,  [A]),
    "TileLang": bench(k,                [A]),
}

bytes_rw = N * M * 4 + N * 4
for name, ms in results.items():
    print(f"  {name:10s}: {ms:.4f} ms  ({bytes_rw / ms / 1e9:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Reduce Sum Latency  (N={N}, M={M}, float32)")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [bytes_rw / ms / 1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Reduce Sum Bandwidth  (N={N}, M={M}, float32)")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()